# WorldSplat-Prune - label lifting to 3D Gaussians (spec 3)

Lifts the per-view label maps produced by the front-end (§2.2,
`Test_Frontend.ipynb`) onto the Gaussians of an **already-trained** model and
builds the per-Gaussian **vote bundle** the pruning operator (§4-5) consumes.

| stage | module | output |
|---|---|---|
| 1 | `scene.Scene` + `GaussianModel.load_ply` | trained Gaussians + ordered views |
| 2 | `datasets` adapter + `semantic_store.SemanticStore` | per-view `L_v` / `dt` on device |
| 3 | `vote.vote_classes_over_views` | `VoteBundle` (`seg`, `purity`, `void_frac`, `boundary_dist`, `n_views`) |
| 4 | `gaussian_renderer` (+ `override_color`) | the lift drawn back on screen |

This is the post-hoc recomputation of the training-time online vote
(`add_semantic_stats`), whose buffers are never persisted to the PLY (§3). It is
**model-side**: unlike `Test_Frontend.ipynb` it needs a trained PLY **and** the
CUDA rasterizer (for the default `render` visibility), plus a scene with labels.

> **Does this work at the current state?** The pass is implemented in
> `worldsplat_prune/vote.py` and its pure paths are CPU-tested
> (`tests/test_vote.py`). The CUDA `render` path is exercised for the first time
> here; everything after section 6 is skippable if you only want the bundle.


## 1. Mount Google Drive

Skip if you run locally / already have the model and dataset on disk.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## 2. Get this repo + build the rasterizer

Colab ships PyTorch. We install `plyfile`/`opencv-python` (and
`transformers`/`timm` only if you also re-run the front-end) and build the two
CUDA deps: `diff-gaussian-rasterization` (rendering) and `simple-knn` (imported
by `scene.gaussian_model`). The superproject records no submodule gitlinks, so
section 2 clones them directly, pins the rasterizer to the fork's `dr_aa`
branch (the renderer always passes `antialiasing=`, even though it is off by
default - a rasterizer built from `main` fails with `unexpected keyword
argument 'antialiasing'`), populates its vendored `glm` submodule, and runs
`patch_cuda_submodules.sh` to add the `<cstdint>` include modern toolchains
dropped.


In [ ]:
import os

REPO_URL = "https://github.com/satyaba/worldsplat-prune.git"
REPO_DIR = "/content/worldsplat-prune"
BRANCH = "main"

if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch --all -q
    !git -C {REPO_DIR} checkout {BRANCH} -q
    !git -C {REPO_DIR} pull --ff-only -q

os.chdir(REPO_DIR)

import subprocess, shutil

RASTERIZER_URL = "https://github.com/graphdeco-inria/diff-gaussian-rasterization.git"
KNN_URL = "https://gitlab.inria.fr/bkerbl/simple-knn.git"


def _git(args):
    subprocess.run(["git"] + args, check=True)


def ensure_clone(path, url, ref=None):
    # The superproject records no submodule gitlinks, so `git submodule update`
    # is a no-op here; clone the two deps the build needs directly.
    if not os.path.exists(os.path.join(path, ".git")):
        shutil.rmtree(path, ignore_errors=True)
        _git(["clone", "-q", url, path])
    if ref is not None:
        _git(["-C", path, "fetch", "-q", "origin", ref])
        _git(["-C", path, "checkout", "-q", "FETCH_HEAD"])
    # diff-gaussian-rasterization vendors glm at third_party/glm: without it the
    # build dies and pip only reports 'No available output'. Populate nested deps.
    _git(["-C", path, "submodule", "update", "--init", "--recursive"])


# The renderer always passes `antialiasing=`, so the rasterizer must be the
# fork's `dr_aa` branch (plain `main` lacks the parameter -> TypeError).
ensure_clone("submodules/diff-gaussian-rasterization", RASTERIZER_URL, ref="dr_aa")
ensure_clone("submodules/simple-knn", KNN_URL)          # default branch `main`

!bash scripts/patch_cuda_submodules.sh
!pip install -q plyfile opencv-python joblib
!pip install -q ./submodules/diff-gaussian-rasterization --no-build-isolation
!pip install -q ./submodules/simple-knn --no-build-isolation

print("cwd:", os.getcwd())

## 3. GPU check

`render` visibility needs the CUDA rasterizer; `project` visibility is pure
torch and CPU-ok (set `VISIBILITY = "project"` in section 4 to skip rendering).


In [ ]:
!nvidia-smi

import torch
print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 4. Configure paths

`MODEL_DIR` follows the training convention `$HOME_DIR/output/<run_id>`
(e.g. `base_30k` from `Run_Experiments.ipynb`). `ITERATION = -1` picks the
highest `point_cloud/iteration_*`.


In [ ]:
import os
from pathlib import Path

HOME_DIR = "/content/drive/MyDrive/Magister/Thesis/Script"
DATASET_NAME = "office_0/Sequence_1"          # the scene the model was trained on
MODEL_NAME = "base_30k"                        # -> $HOME_DIR/output/<MODEL_NAME>
ITERATION = -1                                 # -1 == latest iteration_*

os.environ["HOME_DIR"] = HOME_DIR
DATASET_ROOT = Path(HOME_DIR) / "Dataset"
os.environ["DATASET_ROOT"] = str(DATASET_ROOT)

CANDIDATES = [
    DATASET_ROOT / DATASET_NAME,                        # prepared
    Path(HOME_DIR) / "raw" / Path(DATASET_NAME).name,   # raw retrieval
    Path(DATASET_NAME),                                 # explicit path
]
DATASET_DIR = next((c for c in CANDIDATES if (c / "images").is_dir()), None)
MODEL_DIR = Path(HOME_DIR) / "output" / MODEL_NAME

assert DATASET_DIR is not None, "no 'images/' dir found - edit DATASET_NAME"
assert MODEL_DIR.exists(), f"trained model not found: {MODEL_DIR}"

print("DATASET_DIR:", DATASET_DIR)
print("MODEL_DIR  :", MODEL_DIR)

# --- vote pass knobs -------------------------------------------------------
LABEL_ADAPTER = "composite"   # "composite"/"grounded_sam2" or "replica" for GT
LABEL_DIR = None              # None -> adapter default (semantic_composite)
MIN_AREA = 0                  # drop per-view ids smaller than N px
VISIBILITY = "render"         # "render" (default, CUDA) or "project" (cheap)
VOTE = "first"                # "first" (default) or "majority" (robust to stray views)
DT_NORM = 100.0               # px scaling: dt_norm px from a boundary -> d = 1
MIN_VIEWS = 3                 # below this a Gaussian finalizes to void
OCCLUSION = True              # depth test: ignore labels behind the visible surface
DEPTH_TOL = 0.02              # relative inverse-depth tolerance for the test
MASK_MARGIN = 2.0             # ignore samples within N px of a mask boundary (0 = off)
print("adapter:", LABEL_ADAPTER, "| visibility:", VISIBILITY,
      "| vote:", VOTE, "| occlusion:", OCCLUSION, "| mask_margin:", MASK_MARGIN)


## 5. Load the trained model + scene

Reuses the model's `cfg_args` (as `eval_worldsplat.py` does) so resolution,
background and eval split match training. The vote may see **train cameras
only** ([D10]) so no label leaks into the test split.


In [ ]:
from argparse import ArgumentParser, Namespace
from pathlib import Path

from arguments import ModelParams
from eval_worldsplat import _fill_model_defaults
from scene import Scene
from scene.gaussian_model import GaussianModel

# every ModelParams field, then overlay the model's saved cfg_args
_parser = ArgumentParser()
_mp = ModelParams(_parser, sentinel=True)
cfg = vars(_parser.parse_args([]))
_cfg_path = Path(MODEL_DIR) / "cfg_args"
if _cfg_path.exists():
    cfg.update(vars(eval(_cfg_path.read_text())))
cfg["source_path"] = str(DATASET_DIR)
cfg["model_path"] = str(MODEL_DIR)
dataset = _mp.extract(Namespace(**cfg))
dataset.depths = ""                       # HLOC/COLMAP scenes have no depth
_fill_model_defaults(dataset)

print("source:", dataset.source_path)
print("model :", dataset.model_path,
      "| resolution", dataset.resolution,
      "| white_bg", dataset.white_background)

gaussians = GaussianModel(dataset.sh_degree)
scene = Scene(dataset, gaussians, load_iteration=ITERATION, shuffle=False)
cams = scene.getTrainCameras()
print("gaussians:", gaussians.get_xyz.shape[0], "| train cameras:", len(cams),
      "| iteration:", scene.loaded_iter)

## 6. Semantic store

Wraps the label adapter and serves `{label, dt, normal}` per view at the
camera's training resolution. For the front-end output use adapter
`composite`; for Replica GT use `replica`.


In [ ]:
from worldsplat_prune.datasets import get_adapter
from worldsplat_prune.semantic_store import SemanticStore

adapter = get_adapter(LABEL_ADAPTER, DATASET_DIR,
                      {"label_dir": LABEL_DIR, "min_area": MIN_AREA})
store = SemanticStore(adapter, device="cuda" if torch.cuda.is_available() else "cpu")
print(adapter.meta())

## 7. The vote pass

`vote_classes_over_views` projects each Gaussian centre through every view's
`full_proj_transform`, samples `L_v`/`dt`, and reduces the votes. `render`
membership uses the rasterizer's `radii > 0` (the trainer's convention [D6]);
`project` is the cheap in-frame test.

**Occlusion (`OCCLUSION = True`).** A centre-projection test is depth-blind, so
without a depth test geometry *behind* an object inherits its label and paints a
ghost of the object onto the background (spec Risk 1). The pass drops any sample
whose Gaussian sits behind the rendered surface: its inverse depth `1/z` must be
within `DEPTH_TOL` of the pixel's reference, which is the rasterizer's depth
under `render` or a per-pixel nearest-`z` z-buffer of the centres under `project`.

**Mask rim (`MASK_MARGIN`).** Front-end silhouettes are a pixel or two generous,
so *at rim pixels the front surface is really background*: occlusion passes and
background Gaussians still inherit the object id. `MASK_MARGIN = 2` ignores
samples within that many px of a label boundary (via `dt`), leaving the clean
interiors. Set `0` to disable; raise it if thin slivers still bleed.

**Vote (`VOTE`).** `first` freezes whatever id a Gaussian first sees, so a single
stray view (a rim/gap slip) sticks forever; `majority` needs a plurality and
sends ties to void, which is more robust once occlusion + rim filtering are on.


In [ ]:
from worldsplat_prune.vote import vote_classes_over_views

bundle = vote_classes_over_views(
    gaussians, cams, store,
    visibility=VISIBILITY,
    vote=VOTE,
    dt_norm=DT_NORM,
    min_views=MIN_VIEWS,
    occlusion=OCCLUSION,                  # depth test: drops background ghosts
    depth_tol=DEPTH_TOL,
    mask_margin=MASK_MARGIN,              # drop samples near mask boundaries
    dense_hist=(VOTE == "majority"),      # keep the id histogram for inspection
    camera_order="sorted",                # deterministic view order (first-wins)
)

print(bundle.summary())
print("views processed:", bundle.meta["n_views_processed"], "of", len(cams))
print("samples dropped: occluded =", bundle.meta["n_occluded"],
      "| mask rim =", bundle.meta["n_masked"])
print("view order (first 5):", bundle.meta["view_order"][:5])


## 8. Diagnostics

The one-line summary above is the "is this labelling usable at all" readout.
Here we look at the distributions: support (`n_views`), purity, void fraction and
boundary distance - plus a per-class Gaussian census.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

seg = bundle.seg.cpu().numpy()
purity = bundle.purity.cpu().numpy()
void_frac = bundle.void_frac.cpu().numpy()
bd = bundle.boundary_dist.cpu().numpy()
n_views = bundle.n_views.cpu().numpy()
seen = n_views > 0

fig, ax = plt.subplots(1, 4, figsize=(20, 4))
ax[0].hist(n_views[seen], bins=40); ax[0].set_title("n_views (seen)")
ax[1].hist(purity[seg > 0], bins=40); ax[1].set_title("purity (labeled)")
ax[2].hist(void_frac[seen], bins=40); ax[2].set_title("void_frac (seen)")
ax[3].hist(bd, bins=40); ax[3].set_title("boundary_dist")
plt.tight_layout(); plt.show()

print(f"{seen.sum():,} seen / {seg.size:,} gaussians | {(seg > 0).sum():,} labeled")
print(f"boundary band (d < 0.04): {(bd < 0.04).sum():,} gaussians would be protected")

if LABEL_ADAPTER in ("composite", "grounded_sam2"):
    from worldsplat_prune.datasets.composite import decode
    cls, inst = decode(seg)
    objs = np.unique(seg[seg > 0])
    print(f"distinct objects lifted: {objs.size}")
    from collections import Counter
    print("gaussians per class:", dict(sorted(Counter(cls[cls > 0].tolist()).items())))

## 9. Visualise the lift on screen

For a few train views we show three aligned rows: the trained **RGB**, the
**input label map** the lift reads (`L_v` from the adapter, drawn with the same
id->colour mapping), and the **lifted `seg`** re-rendered with `override_color`.
Matching object colours/silhouettes across the last two rows is the quickest
sanity check that the projection is aligned.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from argparse import ArgumentParser
from arguments import PipelineParams
from gaussian_renderer import render

_p = ArgumentParser()
pipe = PipelineParams(_p).extract(_p.parse_args([]))
bg = torch.tensor([1, 1, 1] if dataset.white_background else [0, 0, 0],
                  dtype=torch.float32, device="cuda")

cmap = plt.get_cmap("nipy_spectral")
base = np.array([0.15, 0.15, 0.15], dtype=np.float32)     # unlabeled -> gray


def colorize_ids(ids):
    """Stable per-id colour, so the input map and the render share a palette."""
    out = np.tile(base, ids.shape + (1,))
    u = np.unique(ids[ids > 0])
    if u.size:
        key = (u.astype(np.int64) % 251) / 251.0
        for i, o in enumerate(u):
            out[ids == o] = cmap(key[i])[:3]
    return out


color_t = torch.tensor(colorize_ids(seg), dtype=torch.float32, device="cuda")

show = cams[:3]
with torch.no_grad():
    fig, axes = plt.subplots(3, len(show), figsize=(6 * len(show), 13),
                             squeeze=False)
    for j, cam in enumerate(show):
        H, W = int(cam.image_height), int(cam.image_width)

        # the input label map this view contributes to the vote
        sem = store.get(cam.image_name, H, W)
        if sem is not None:
            lab = sem["label"].detach().cpu().numpy()
            axes[1, j].imshow(colorize_ids(lab))
        else:
            axes[1, j].text(0.5, 0.5, "no label map", ha="center", va="center")

        rgb = render(cam, gaussians, pipe, bg)["render"].clamp(0, 1)
        labelled = render(cam, gaussians, pipe, bg,
                          override_color=color_t)["render"].clamp(0, 1)

        axes[0, j].imshow(rgb.permute(1, 2, 0).cpu())
        axes[0, j].set_title("RGB - " + cam.image_name)
        axes[1, j].set_title("input map (L_v)")
        axes[2, j].imshow(labelled.permute(1, 2, 0).cpu())
        axes[2, j].set_title("lifted seg")
        for a in axes[:, j]:
            a.axis("off")
    plt.tight_layout(); plt.show()


## 10. 3D sanity + next steps

A subsampled 3D scatter coloured by `seg` (and by `purity`, to spot ambiguous
floaters). Then the bundle feeds the pruning operator (§4-5).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

xyz = gaussians.get_xyz.detach().cpu().numpy()
rng = np.random.default_rng(0)
idx = rng.choice(xyz.shape[0], size=min(20_000, xyz.shape[0]), replace=False)

fig = plt.figure(figsize=(14, 6))
for k, (vals, title) in enumerate([(colors[idx], "seg"),
                                   (1.0 - purity[idx], "1 - purity")]):
    ax = fig.add_subplot(1, 2, k + 1, projection="3d")
    if title == "seg":
        ax.scatter(xyz[idx, 0], xyz[idx, 1], xyz[idx, 2], c=vals, s=1)
    else:
        ax.scatter(xyz[idx, 0], xyz[idx, 1], xyz[idx, 2], c=vals,
                   cmap="magma", vmin=0, vmax=1, s=1)
    ax.set_title(title + " (subsample)")
plt.tight_layout(); plt.show()

# The next component is the pruning operator (spec 4-5), which consumes `bundle`
# (seg, purity, void_frac, boundary_dist, n_views) together with the geometry
# (opacity, scaling) and scene.cameras_extent.
print("bundle fields:",
      [k for k in ("seg", "purity", "void_frac", "boundary_dist", "n_views")])

### Troubleshooting

- **`render` import / build fails** -> the CUDA submodules did not build. Check
  the section-2 pip output, or set `VISIBILITY = "project"` in section 4 to run
  the whole vote pass without the rasterizer.
- **Wheel build fails with `No available output`** -> pip is hiding the
  compiler error; it is almost always the vendored `glm` submodule missing
  (`setup.py` builds with `-I third_party/glm/`). Re-run section 2, or
  `!git -C submodules/diff-gaussian-rasterization submodule update --init --recursive`,
  then rebuild. To see the real error:
  `!cd submodules/diff-gaussian-rasterization && python setup.py build_ext --inplace 2>&1 | tail -60`.
- **`GaussianRasterizationSettings ... unexpected keyword argument 'antialiasing'`**
  -> the rasterizer was built from `main` instead of the pinned `dr_aa` branch
  (section 2 now checks it out explicitly). Re-run section 2, or
  `!git -C submodules/diff-gaussian-rasterization checkout FETCH_HEAD && pip install ./submodules/diff-gaussian-rasterization --no-build-isolation`.
- **An object's colour leaks onto the background (a ghost of the object)**
  -> occlusion is off/too loose, or the rasterizer returned no depth. Keep
  `OCCLUSION = True` and lower `DEPTH_TOL`; the `dr_aa` branch returns the
  depth map the test needs.
- **Still a fraction of background Gaussians labelled as the object**
  -> mostly mask-rim slips (rare, but `first` makes them permanent). Raise
  `MASK_MARGIN` (2-4 px), lower `DEPTH_TOL`, and/or set `VOTE = "majority"`
  so a stray view is outvoted. A large `MASK_MARGIN` can starve very small
  objects (their whole mask is 'rim') - keep it a couple of px.
- **`labeled == 0` or very low** -> the label maps did not resolve for the
  training image stems, or `min_views` filtered everything. Check
  `adapter.meta()` (section 6) `num_frames`, and lower `MIN_VIEWS`.
- **Everything void / mismatched colours in section 9** -> the label maps and the
  trained model disagree on the scene; verify they share the same `DATASET_DIR`
  (the model's `cfg_args` `source_path` must equal `DATASET_DIR`).
- **`majority` OOM** -> `majority` keeps the votes to build the histogram; use
  the default `first` (streaming, O(N)) for large scenes.
- **Composite vs GT** -> to validate classes against Replica GT before trusting
  pruning, run `prep.eval_instances --class_gt semantic_class --level class`
  (see `Test_Frontend.ipynb` section 10).
